# 🤖 Notebook 04 — Predictive Modeling
## Spatio-Temporal Air Quality Forecasting — Karachi, Pakistan

This notebook implements and compares three distinct modeling approaches for air quality prediction:
1. **Tree-Based Models (Random Forest / XGBoost):** Best for non-linear relationships and feature importance.
2. **Support Vector Regression (SVR):** Robust to high-dimensional satellite data.
3. **Long Short-Term Memory (LSTM):** Captures temporal dependencies and lag effects.

### 🎯 Evaluation Metrics:
- RMSE (Root Mean Squared Error)
- MAE (Mean Absolute Error)
- R² Score
- Spatial Cross-Validation (Station-wise holdout)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path

# ── Load Processed Data ──────────────────────────────────────────────────────
PROCESSED_DIR = Path('data/processed')
df_raw = pd.read_csv(PROCESSED_DIR / 'karachi_final_raw.csv', parse_dates=['date'])
df_std = pd.read_csv(PROCESSED_DIR / 'karachi_final_std.csv', parse_dates=['date'])
df_mm  = pd.read_csv(PROCESSED_DIR / 'karachi_final_mm.csv', parse_dates=['date'])

print(f"✓ Loaded raw data: {df_raw.shape}")
print(f"✓ Loaded standardized data: {df_std.shape}")
print(f"✓ Loaded min-max data: {df_mm.shape}")

## 1. Feature Importance (Random Forest)
Using the raw dataset (Trees are scale-invariant).

In [ ]:
target = 'aer_ai'  # Using Aerosol Index as proxy for PM
features = [c for c in df_raw.columns if c not in ['date', 'station', 'month', target]]

X = df_raw[features]
y = df_raw[target]

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

fi = pd.DataFrame({'feature': features, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=fi.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Features for Air Quality Prediction')
plt.show()

## 2. Spatial Cross-Validation (Station Holdout)
We evaluate how well the model generalizes to a station it hasn't seen during training.

In [ ]:
gkf = GroupKFold(n_splits=5)
groups = df_raw['station']

rmses = []
r2s = []

for train_idx, val_idx in gkf.split(X, y, groups=groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = RandomForestRegressor(n_estimators=100, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    
    rmses.append(np.sqrt(mean_squared_error(y_val, preds)))
    r2s.append(r2_score(y_val, preds))

print(f"Avg Station-wise RMSE: {np.mean(rmses):.4f}")
print(f"Avg Station-wise R²  : {np.mean(r2s):.4f}")